# 06 Observability and Troubleshooting (OpenClaw, 2026)

## What This Lesson Is
Build reproducible troubleshooting workflows by classifying failures and collecting minimal diagnostics.

## Scientific Lens
- Concept: Failure taxonomy plus targeted diagnostics for faster MTTR.
- Measure: Triage coverage across known error categories.
- Validity Limit: Logs alone may not reveal provider-side incidents.


## How It Works
1. Categorize sample errors into network/auth/config/runtime classes.
2. Generate deterministic triage recommendations from category mapping.
3. Run live gateway diagnostics and capture results.


In [ ]:
import os
import shutil

HAS_OPENCLAW = shutil.which("openclaw") is not None
print("openclaw available:", HAS_OPENCLAW)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("OLLAMA_BASE_URL:", os.getenv("OLLAMA_BASE_URL", "<unset>"))


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: runs real OpenClaw CLI operations when available; otherwise prints explicit skip guidance.


In [ ]:
# Deterministic Demo
samples = [
    "ConnectionRefusedError: [Errno 61] Connection refused",
    "401 Unauthorized: invalid API key",
    "Config validation failed: expected string",
    "Kernel died while waiting for execute reply",
]
def categorize(msg):
    m = msg.lower()
    if "unauthorized" in m or "api key" in m:
        return "auth"
    if "connection" in m or "refused" in m:
        return "network"
    if "config" in m:
        return "config"
    return "runtime"
cats = [categorize(m) for m in samples]
print(cats)
assert set(cats) == {"auth", "network", "config", "runtime"}


In [ ]:
# Live Demo
import shutil
import subprocess

if shutil.which("openclaw") is None:
    print("Skipping live troubleshooting demo: openclaw CLI is not installed.")
else:
    for cmd in (["openclaw", "gateway", "status"], ["openclaw", "doctor"]):
        print("$", " ".join(cmd))
        proc = subprocess.run(cmd, capture_output=True, text=True)
        print((proc.stdout or proc.stderr).strip()[:1500])


## Applied Labs
1. Add parser extracting remediation commands from diagnostic text.
2. Track category frequency over 50 synthetic incidents and report top 2 classes.
3. Create runbook mapping each class to owner and SLO impact.

## Validation Checklist
- Failure category mapping is deterministic and test-backed.
- Triage output provides explicit next actions.
- Live diagnostics degrade gracefully if commands are unavailable.

## Further Reading
- Google SRE incident response: https://sre.google/sre-book/managing-incidents/
- OpenClaw troubleshooting docs: https://docs.openclaw.ai/troubleshooting
- Python error handling: https://docs.python.org/3/tutorial/errors.html
